# Lab 1.3, Build 2 — Build Tina in four stages

**Before you start:** select **Cell > Run All** (or Shift+Enter through each cell in order) to initialize the harness. Then work through the cells marked `# ── YOUR WORK ──`.

**Tina** is Cortex Bank's AI compliance assistant. You build her in four stages, each adding one capability. At each stage, run the provided question and record the output — the check reads your traces.

---
**Your four stages:**
| Stage | You add | Gap |
|---|---|---|
| 1 | Bare completion | No source consulted |
| 2 | Context injection from `cortex-policies` | Cannot choose documents mid-answer |
| 3 | `search_policies` tool + dispatch | One shot only |
| 4 | ReAct loop | (target — no gap for this task) |

Cells marked `# ── YOUR WORK ──` are the ones you edit.

In [ ]:
# ── Harness setup (run once) ──────────────────────────────────────────────────
import sys, os, json, pathlib

sys.path.insert(0, '/opt/ara/lib')
from tina.client import llm_client, es_client, model_fast, model_strong
from tina.tools import ToolRegistry
from tina.loop import react_loop

# Load environment written by challenge 01 setup
env_path = pathlib.Path('/home/elastic/env')
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

client = llm_client()
es = es_client()
FAST  = model_fast()
STRONG = model_strong()

# Read the seeded constraint shown to you in Build 1
constraint_path = pathlib.Path('/home/elastic/constraint.json')
constraint = json.loads(constraint_path.read_text()) if constraint_path.exists() else {}

RESULTS = pathlib.Path('/home/elastic/.traces')
RESULTS.mkdir(parents=True, exist_ok=True)

print('Tina harness ready.')
print(f'  FAST model:   {FAST}')
print(f'  STRONG model: {STRONG}')
if constraint:
    print(f'  Your constraint: {constraint.get("label", "(none)")}')  

---
## Stage 1 — Bare completion

Call the LLM directly. No retrieval. No tools. Record what Tina says — the check verifies she did **not** cite the correct $10,000 CTR threshold (if she guesses right, it records that too, but the trace will show no retrieval).

In [ ]:
# Stage 1 scaffolding — do not edit
QUESTION_SINGLE = (
    "What is Cortex Bank's CTR threshold for cash transactions, "
    "and is a $9,500 cash deposit reportable?"
)

SYSTEM_PROMPT_BASE = (
    "You are Tina, Cortex Bank and Trust's compliance assistant. "
    "Answer compliance questions concisely and cite your sources."
)

In [ ]:
# ── YOUR WORK ── Stage 1: bare completion ─────────────────────────────────────
# Call the LLM with QUESTION_SINGLE and SYSTEM_PROMPT_BASE.
# Use FAST or STRONG — your choice here (choose based on your Build 1 constraint).
# Store the response text in: stage1_answer

# Example shape (replace with your call):
# response = client.chat.completions.create(
#     model=FAST,
#     messages=[
#         {"role": "system", "content": SYSTEM_PROMPT_BASE},
#         {"role": "user",   "content": QUESTION_SINGLE},
#     ],
#     temperature=0,
# )
# stage1_answer = response.choices[0].message.content

stage1_answer = ""  # ← replace with your actual call

In [ ]:
# Record stage 1 (run after you fill in stage1_answer)
assert stage1_answer, "Run your Stage 1 call first and assign stage1_answer."
s1_trace = {"stage": 1, "question": QUESTION_SINGLE, "answer": stage1_answer, "tool_calls": []}
(RESULTS / 'stage-1-trace.json').write_text(json.dumps(s1_trace, indent=2))
print("Stage 1 recorded. Tina says:")
print(stage1_answer[:300])

---
## Stage 2 — Context injection

Before calling the LLM, retrieve the top-3 chunks from `cortex-policies` and inject them into the prompt. The check verifies that Tina's answer now contains the correct $10,000 threshold.

In [ ]:
# ── YOUR WORK ── Stage 2: context injection ───────────────────────────────────
# 1. Write a semantic search query against the cortex-policies index.
#    Use the Elasticsearch client (es) with the semantic field 'body_semantic'.
#    Retrieve k=3 results.
# 2. Format the retrieved chunks as context.
# 3. Build a new prompt that includes the context and the question.
# 4. Call the LLM and store the result in: stage2_answer
# 5. Store the retrieved document ids in: stage2_retrieved_ids

# Hints:
# resp = es.search(index='cortex-policies', body={...}, size=3)
# hits = resp['hits']['hits']
# context = '\n\n'.join(h['_source']['body'] for h in hits)
# stage2_retrieved_ids = [h['_id'] for h in hits]

stage2_answer = ""       # ← your answer after context injection
stage2_retrieved_ids = []  # ← list of retrieved _id values

In [ ]:
assert stage2_answer, "Run your Stage 2 call first."
s2_trace = {"stage": 2, "question": QUESTION_SINGLE, "answer": stage2_answer,
            "retrieved_ids": stage2_retrieved_ids, "tool_calls": []}
(RESULTS / 'stage-2-trace.json').write_text(json.dumps(s2_trace, indent=2))
print("Stage 2 recorded. Tina says:")
print(stage2_answer[:300])

---
## Stage 3 — Tool calling

Register `search_policies` as a tool the LLM can call. Write the JSON schema and the dispatch function. The check verifies that the trace shows one `search_policies` call with a query mentioning the topic.

In [ ]:
# ── YOUR WORK ── Stage 3: tool calling ───────────────────────────────────────
# Define the search_policies tool.
# Use the ToolRegistry from the tina harness.

tools_s3 = ToolRegistry()

# @tools_s3.register(
#     name="search_policies",
#     description="Search Cortex Bank AML policies and procedures",
#     parameters={
#         "type": "object",
#         "properties": {
#             "query": {"type": "string", "description": "Search query"}
#         },
#         "required": ["query"],
#     }
# )
# def search_policies(query: str) -> list:
#     resp = es.search(index='cortex-policies',
#                      body={"query": {"semantic": {"field": "body_semantic", "query": query}}},
#                      size=3)
#     return [{"policy_id": h['_source'].get('policy_id', h['_id']),
#              "body": h['_source']['body'][:600]} for h in resp['hits']['hits']]

# Call the model with the tool (single turn — no loop yet)
# response = client.chat.completions.create(
#     model=FAST,
#     messages=[{"role": "system", "content": SYSTEM_PROMPT_BASE},
#               {"role": "user",   "content": QUESTION_SINGLE}],
#     tools=tools_s3.schema(),
#     tool_choice="auto",
#     temperature=0,
# )
# Stage 3 records the tool call (not the final answer — the loop isn't complete yet)

stage3_tool_calls = []  # ← list of {name, arguments} dicts from the response

In [ ]:
assert stage3_tool_calls, "Complete Stage 3 and capture stage3_tool_calls."
s3_trace = {"stage": 3, "question": QUESTION_SINGLE, "tool_calls": stage3_tool_calls}
(RESULTS / 'stage-3-trace.json').write_text(json.dumps(s3_trace, indent=2))
print(f"Stage 3 recorded. Tool calls: {[tc['name'] for tc in stage3_tool_calls]}")

---
## Stage 4 — ReAct loop

Complete the loop: handle `tool_calls`, dispatch, append `role: tool` results, continue until `finish_reason: stop`. Then run the **two-hop question** that requires searching for *both* the CTR threshold and the structuring red flags — two different policies.

In [ ]:
# Stage 4 scaffolding
QUESTION_TWO_HOP = (
    "What is Cortex Bank's CTR threshold, "
    "and what are the main structuring red flags in the AML policy?"
)

tools_s4 = ToolRegistry()
# Register search_policies again for the loop (copy your Stage 3 implementation below)

In [ ]:
# ── YOUR WORK ── Stage 4: ReAct loop ─────────────────────────────────────────
# 1. Register search_policies on tools_s4 (same as Stage 3).
# 2. Complete the loop below. The scaffold shows the structure;
#    fill in the two lines marked TODO.

# --- Register your tool here ---

# --- Loop ---
def run_react_loop(user_question: str, model: str, max_iter: int = 8) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_BASE},
        {"role": "user",   "content": user_question},
    ]
    all_tool_calls = []
    for _ in range(max_iter):
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools_s4.schema(),
            tool_choice="auto",
            temperature=0,
        )
        msg = response.choices[0].message
        if msg.tool_calls:
            # TODO: append the assistant message with tool_calls to messages
            # TODO: dispatch each tool call, append role:tool message
            for tc in msg.tool_calls:
                all_tool_calls.append({"name": tc.function.name, "arguments": tc.function.arguments})
        else:
            return {"answer": msg.content, "tool_calls": all_tool_calls}
    return {"answer": "(max iterations)", "tool_calls": all_tool_calls}

# Run the two-hop question
stage4_result = {}  # ← replace with: run_react_loop(QUESTION_TWO_HOP, FAST)

In [ ]:
assert stage4_result.get('answer'), "Complete Stage 4 and assign stage4_result."
s4_trace = {"stage": 4, "question": QUESTION_TWO_HOP,
            "answer": stage4_result.get('answer', ''),
            "tool_calls": stage4_result.get('tool_calls', [])}
(RESULTS / 'stage-4-trace.json').write_text(json.dumps(s4_trace, indent=2))
print(f"Stage 4 recorded.")
print(f"  Tool calls: {len(s4_trace['tool_calls'])} (need at least 2 with different queries)")
print(f"  Answer preview: {s4_trace['answer'][:200]}")

---
## Submit

When all four stages are recorded, run the cell below to confirm, then select **Check** in the sidebar.

In [ ]:
import pathlib, json
traces_dir = pathlib.Path('/home/elastic/.traces')
expected = ['stage-1-trace.json', 'stage-2-trace.json', 'stage-3-trace.json', 'stage-4-trace.json']
missing = [f for f in expected if not (traces_dir / f).exists()]
if missing:
    print(f'Missing traces: {missing}. Run all stage record cells first.')
else:
    print('All four stage traces recorded. Select Check in the sidebar.')